In [ ]:
# =============================================
# Step 1: Mount Google Drive
# =============================================
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
cd /content/drive/MyDrive/

/content/drive/MyDrive


In [ ]:
!pip uninstall -y transformers
!pip install transformers==4.38.2 --quiet


Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.7/130.7 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 104.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 65.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 169.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.3.0 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.38.2 which is incompatible.


In [ ]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm
import numpy as np

# Load file
df = pd.read_csv("all_data_after2_updated.csv")

# Column indices
#categorical_indices = [1, 2, 3, 4, 8, 9, 15, 19, 21, 22, 23, 24, 25, 27, 29, 31, 35, 36, 37, 41]
#numerical_indices = [10, 11, 12, 13, 14, 16, 17, 18, 20, 26, 32, 33, 34, 39, 40]
textual_columns = ['Event full description', 'Lesson Learnt', 'Post-event summary']

# Column names
#categorical_indices_raw = [1, 2, 3,  8,  15,  21, 22, 23,  27, 29, 31, 35, 36, 37]
categorical_indices_raw = [1, 2, 3, 8, 21, 22, 23,  27, 29, 31, 35, 36, 37]

#numerical_indices_raw = [10, 11, 12, 13, 14, 16, 17, 18, 20, 26, 32, 33, 34, 39, 40]
numerical_indices_raw = [10,  13, 14, 20,  32, 33, 34, 39, 40]
categorical_columns = [df.columns[i] for i in categorical_indices_raw]
numerical_columns = [df.columns[i] for i in numerical_indices_raw]

# Exactly 38 total attributes
modalities = ['txt'] * 3 + ['cat'] * 13 + ['num'] * 9
# Load MiniLM model
tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")
model = AutoModel.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")
model.eval()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 384, padding_idx=0)
    (position_embeddings): Embedding(512, 384)
    (token_type_embeddings): Embedding(2, 384)
    (LayerNorm): LayerNorm((384,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-5): 6 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=384, out_features=384, bias=True)
            (key): Linear(in_features=384, out_features=384, bias=True)
            (value): Linear(in_features=384, out_features=384, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=384, out_features=384, bias=True)
            (LayerNorm): LayerNorm((384,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
    

In [ ]:
import torch
import torch.nn as nn


class RAGNetProjectionModule(nn.Module):
    def __init__(self, d_llm=384, d_num=32, d_cat=16, d_shared=384,
                 num_numerical_attrs=15, num_categorical_attrs=20,
                 num_classes_per_cat_attr=None):
        super().__init__()

        if num_classes_per_cat_attr is None:
            num_classes_per_cat_attr = [10] * num_categorical_attrs

        # Numerical value encoders
        self.num_proj = nn.ModuleList([
            nn.Sequential(
                nn.Linear(1, d_num),
                nn.ReLU(),
                nn.LayerNorm(d_num)
            ) for _ in range(num_numerical_attrs)
        ])

        # Categorical class index embeddings
        self.cat_emb = nn.ModuleList([
            nn.Embedding(n_cls, d_cat) for n_cls in num_classes_per_cat_attr
        ])

        # Final projection to shared latent space
        self.shared_proj_num = nn.ModuleList([
            nn.Linear(d_llm + d_num, d_shared) for _ in range(num_numerical_attrs)
        ])

        self.shared_proj_cat = nn.ModuleList([
            nn.Linear(d_llm + d_cat, d_shared) for _ in range(num_categorical_attrs)
        ])

    def forward_numerical(self, attr_idx, llm_emb, raw_value):
        """
        llm_emb:   (d_llm,) or (1, d_llm)
        raw_value: scalar tensor / python float / numpy scalar
        """
        proj_layer = self.num_proj[attr_idx]
        device = proj_layer[0].weight.device

        llm_emb = llm_emb.to(device)
        if llm_emb.dim() == 1:
            llm_emb = llm_emb.unsqueeze(0)  # (1, d_llm)

        if not isinstance(raw_value, torch.Tensor):
            raw_value = torch.tensor(raw_value, dtype=torch.float32, device=device)
        else:
            raw_value = raw_value.to(device=device, dtype=torch.float32)

        if raw_value.dim() == 0:
            raw_value = raw_value.unsqueeze(0).unsqueeze(-1)   # (1, 1)
        elif raw_value.dim() == 1:
            raw_value = raw_value.unsqueeze(-1)                # (B, 1)

        num_vector = self.num_proj[attr_idx](raw_value)        # (B, d_num)
        concat = torch.cat([llm_emb, num_vector], dim=-1)      # (B, d_llm + d_num)
        return self.shared_proj_num[attr_idx](concat).squeeze(0)  # (d_shared,)

    def forward_categorical(self, attr_idx, llm_emb, class_index):
        """
        llm_emb:     (d_llm,) or (B, d_llm)
        class_index: scalar or (B,)
        """
        emb_layer = self.cat_emb[attr_idx]
        device = emb_layer.weight.device

        llm_emb = llm_emb.to(device)
        if llm_emb.dim() == 1:
            llm_emb = llm_emb.unsqueeze(0)  # (1, d_llm)

        if not isinstance(class_index, torch.Tensor):
            class_index = torch.tensor([class_index], dtype=torch.long, device=device)
        else:
            class_index = class_index.to(device=device, dtype=torch.long).view(-1)

        max_index = emb_layer.num_embeddings - 1
        class_index = class_index.clamp(min=0, max=max_index)

        cat_vector = emb_layer(class_index)                    # (B, d_cat)
        concat = torch.cat([llm_emb, cat_vector], dim=-1)     # (B, d_llm + d_cat)
        return self.shared_proj_cat[attr_idx](concat)         # (B, d_shared)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class CrossModalGNNLayer(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.d = d

        # Local GNN weights
        self.W_g = nn.Linear(d, d, bias=False)
        self.a = nn.Parameter(torch.randn(2 * d))

        # Cross-modal attention weights
        self.W_c = nn.Linear(d, d, bias=False)
        self.W_q = nn.Linear(d, d, bias=False)
        self.W_k = nn.Linear(d, d, bias=False)

        # Output activation
        self.activation = nn.GELU()

    def forward(self, Z, edge_index):
        """
        Parameters:
            Z: (M, d) node embeddings for all attributes
            edge_index: (2, E) tensor for edges in COO format (source, target)

        Returns:
            Updated node embeddings (M, d)
        """
        M, d = Z.size()
        #print(M)
        #print(d)
        WgZ = self.W_g(Z)  # (M, d)
        alpha = torch.zeros(M, M, device=Z.device)
        alpha = alpha.to(Z.device)
        edge_index = edge_index.to(Z.device)


        # Compute local attention weights α_ij
        for idx in range(edge_index.size(1)):
            i, j = edge_index[0, idx], edge_index[1, idx]
            cat_ij = torch.cat([WgZ[i], WgZ[j]], dim=-1)
            alpha[i, j] = F.leaky_relu(torch.dot(self.a, cat_ij))

        for i in range(M):
            neighbors = edge_index[1][edge_index[0] == i]
            if len(neighbors) > 0:
                alpha[i, neighbors] = F.softmax(alpha[i, neighbors], dim=0)

        local_msg = torch.matmul(alpha, WgZ)

        # Cross-modal attention β_ij
        Q = self.W_q(Z)  # (M, d)
        K = self.W_k(Z)  # (M, d)
        attn_scores = torch.matmul(Q, K.T) / (d ** 0.5)
        beta = F.softmax(attn_scores, dim=-1)
        global_msg = torch.matmul(beta, self.W_c(Z))  # (M, d)

        #H = self.activation(global_msg)#(local_msg + global_msg)
        H = F.leaky_relu(local_msg + global_msg+Z)
        #H=local_msg
        return H


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class PerAttributeMemoryBank:
    def __init__(self, num_attributes, d_h, max_memory=300):
        self.num_attributes = num_attributes
        self.d_h = d_h
        self.max_memory = max_memory
        self.bank = {i: [] for i in range(num_attributes)}

    def add(self, attr_id, embedding, modality, value=None):
        """
        Store memory for an attribute.

        Args:
            attr_id (int): attribute index
            embedding (Tensor): shape (d_h,)
            modality (str): one of {"txt", "num", "cat"}
            value: raw value for num/cat modalities
        """
        item = {
            "embedding": embedding.detach().cpu(),
            "modality": modality
        }

        if modality in {"num", "cat"}:
            item["value"] = value

        self.bank[attr_id].append(item)

        if len(self.bank[attr_id]) > self.max_memory:
            self.bank[attr_id].pop(0)

    def get(self, attr_id, return_values=False):
        """
        Retrieve memory for an attribute.

        Args:
            attr_id (int): attribute index
            return_values (bool): if True, return both embeddings and values

        Returns:
            embeddings if return_values=False
            (embeddings, values) if return_values=True
        """
        if attr_id not in self.bank or len(self.bank[attr_id]) == 0:
            return (None, None) if return_values else None

        items = self.bank[attr_id]
        embeddings = torch.stack([x["embedding"] for x in items], dim=0)

        if not return_values:
            return embeddings

        values = []
        for x in items:
            if x["modality"] in {"num", "cat"}:
                values.append(x.get("value", None))
            else:
                values.append(None)

        return embeddings, values


class AttributeMemoryFusion(nn.Module):
    def __init__(self, d_h, memory_bank):
        super(AttributeMemoryFusion, self).__init__()
        self.d_h = d_h
        self.memory_bank = memory_bank
        self.W_g = nn.Linear(d_h, d_h)
        self.U_g = nn.Linear(d_h, d_h)
        self.b_g = nn.Parameter(torch.zeros(d_h))

    def forward(self, h_tilde, return_memory_values=False):
        """
        Args:
            h_tilde: Tensor of shape (M, d_h)
            return_memory_values: if True, also return stored values

        Returns:
            u if return_memory_values=False
            (u, retrieved_values) if return_memory_values=True
        """
        M = h_tilde.size(0)
        u = torch.zeros_like(h_tilde)
        retrieved_values = {}

        device = h_tilde.device

        for i in range(M):
            #if return_memory_values:
            mem, values = self.memory_bank.get(i, return_values=True)
             #   retrieved_values[i] = values
            #else:
            #    mem = self.memory_bank.get(i)

            if mem is None or mem.size(0) == 0:
                u[i] = h_tilde[i]
                continue

            mem = mem.to(device)

            # Filter only non-zero memory vectors
            non_zero_mask = mem.abs().sum(dim=1) > 1e-6
            mem = mem[non_zero_mask]

            if mem.size(0) == 0:
                u[i] = h_tilde[i]
                continue

            if return_memory_values:
                retrieved_values[i] = [
                    v for j, v in enumerate(retrieved_values[i]) if non_zero_mask[j].item()
                ]

            # Attention-based fusion using embeddings only
            sim_scores = torch.matmul(mem, h_tilde[i])      # (N,)
            attn_weights = F.softmax(sim_scores, dim=0)     # (N,)
            r_i = torch.sum(attn_weights[:, None] * mem, dim=0)  # (d_h,)

            g_i = torch.sigmoid(self.W_g(h_tilde[i]) + self.U_g(r_i) + self.b_g)
            u[i] = g_i * r_i + (1 - g_i) * h_tilde[i]

        #if return_memory_values:
        #    return u, retrieved_values

        return u

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class NumericalHead(nn.Module):
    def __init__(self, d_h):
        super(NumericalHead, self).__init__()
        self.reg_head = nn.Linear(d_h, 1)

    def forward(self, u):
        return self.reg_head(u).squeeze(-1)  # Scalar output per node
import torch.nn as nn
import torch.nn.functional as F

class NumericalHead(nn.Module):
    def __init__(self, d_h):
        super(NumericalHead, self).__init__()
        self.fc1 = nn.Linear(d_h, d_h // 2)
        self.fc2 = nn.Linear(d_h // 2, 1)
        self.dropout = nn.Dropout(p=0.1)
        self.norm = nn.LayerNorm(d_h // 2)

    def forward(self, u):
        x = F.relu(self.fc1(u))
        x = self.norm(x)
        x = self.dropout(x)
        return self.fc2(x).squeeze(-1)  # Scalar output per node


class CategoricalHead(nn.Module):
    def __init__(self, d_h, num_classes):
        super(CategoricalHead, self).__init__()
        self.cls_head = nn.Linear(d_h, num_classes)

    def forward(self, u):
        return F.softmax(self.cls_head(u), dim=-1)  # Class probabilities


import torch
import torch.nn as nn
import torch.nn.functional as F

class TextualDecoder(nn.Module):
 def __init__(self, d_h, vocab_size, max_len=64, decoder_dim=384, nhead=8, num_layers=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, decoder_dim)
        self.pos_embedding = nn.Parameter(torch.randn(1, max_len, decoder_dim))
        self.fc_in = nn.Linear(d_h, decoder_dim)

        decoder_layer = nn.TransformerDecoderLayer(d_model=decoder_dim, nhead=nhead, batch_first=True)
        self.transformer_decoder = nn.TransformerDecoder(decoder_layer, num_layers=num_layers)

        self.out_proj = nn.Linear(decoder_dim, vocab_size)
        self.max_len = max_len
        self.init_token = 0  # BOS token

 def forward(self, u, target=None, teacher_forcing=True):
    """
    Args:
        u: Tensor of shape (B, decoder_dim) or (decoder_dim,)
        target: LongTensor of shape (B, T) or (T,)
        teacher_forcing: bool — whether to use teacher forcing or inference loop
    Returns:
        logits or generated token ids
    """
    if u.dim() == 1:
        u = u.unsqueeze(0)  # Make (1, d)
    batch_size = u.size(0)

    u_proj = self.fc_in(u)  # (B, decoder_dim)

    if teacher_forcing and target is not None:
        if target.dim() == 1:
            target = target.unsqueeze(0)  # (1, T)

        if target.max().item() >= self.embedding.num_embeddings:
            print(f"⚠️ Token index {target.max().item()} exceeds vocab size {self.embedding.num_embeddings}")
            target = target.clamp(max=self.embedding.num_embeddings - 1)

        tgt_emb = self.embedding(target)  # (B, T, D)
        if not hasattr(self, "pos_embedding"):
            self.pos_embedding = nn.Parameter(torch.zeros(1, self.max_len, self.embedding.embedding_dim))

        tgt_emb = tgt_emb + self.pos_embedding[:, :tgt_emb.size(1), :]  # Positional + token embedding
        memory = u_proj.unsqueeze(1).expand(-1, tgt_emb.size(1), -1)    # (B, T, D)
        out = self.transformer_decoder(tgt=tgt_emb, memory=memory)      # (B, T, D)
        logits = self.out_proj(out)                                     # (B, T, vocab_size)
        return logits

    # --- Inference (no teacher forcing) ---
    input_token = torch.full((batch_size, 1), self.init_token, dtype=torch.long, device=u.device)  # (B, 1)
    generated = []

    for t in range(self.max_len):
      tgt_emb = self.embedding(input_token)
      tgt_emb = tgt_emb + self.pos_embedding[:, :tgt_emb.size(1), :]

      memory = u_proj.unsqueeze(1).expand(-1, tgt_emb.size(1), -1)
      out = self.transformer_decoder(tgt=tgt_emb, memory=memory)
      logits = self.out_proj(out[:, -1:, :])  # (B, 1, vocab)
      next_token = logits.argmax(dim=-1)     # (B, 1)

      input_token = torch.cat([input_token, next_token], dim=1)  # (B, t+2)
      generated.append(next_token)

    # Stop if all sequences predicted EOS
      if (next_token == self.eos_token).all():
        break
      print(generated)

    return torch.cat(generated, dim=1)  # (B, T)




#    def forward(self, u, target=None, teacher_forcing=True):
#        batch_size = u.size(0)
#        hidden = self.fc_in(u).unsqueeze(0)  # (1, batch, hidden_dim)
#        outputs = []

#        if teacher_forcing and target is not None:
#            embedded = self.embedding(target)
#            out, _ = self.rnn(embedded, hidden)
#            logits = self.out_proj(out)  # (batch, seq_len, vocab_size)
#            return logits
#        else:
#            input_token = torch.full((batch_size, 1), self.init_token, dtype=torch.long, device=u.device)
#            for _ in range(self.max_len):
#                embedded = self.embedding(input_token)
#                out, hidden = self.rnn(embedded, hidden)
#                logits = self.out_proj(out[:, -1, :])
#                outputs.append(logits.unsqueeze(1))
#                input_token = logits.argmax(dim=-1, keepdim=True)
#            return torch.cat(outputs, dim=1)  # (batch, max_len, vocab_size)


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from scipy.stats import gaussian_kde


class MARIPLoss(nn.Module):
    def __init__(self, d_h, alpha=0.1, epsilon=1e-6):
        super(MARIPLoss, self).__init__()
        self.alpha = alpha
        self.epsilon = epsilon

        self.kde_dict = {}         # For numerical density
        self.kde_ref = {}          # Reference density for normalization
        self.class_counts = {}     # For categorical frequency
        self.observed_freq = {}    # For missingness prior

        # Modality balancing
        self.num_scale = 0.5
        self.cat_scale = 0.5
        self.txt_scale = 1.0

    def register_numerical_kde(self, attr_idx, values):
        values = np.asarray(values, dtype=np.float64)
        values = values[np.isfinite(values)]
        if len(values) > 1:
            kde = gaussian_kde(values)
            self.kde_dict[attr_idx] = kde

            dens = kde(values)
            dens = dens[np.isfinite(dens)]
            self.kde_ref[attr_idx] = float(np.median(dens)) if len(dens) > 0 else 1.0

    def register_class_counts(self, attr_idx, class_labels):
        class_labels = np.asarray(class_labels)
        unique, counts = np.unique(class_labels, return_counts=True)
        self.class_counts[attr_idx] = dict(zip(unique, counts))

    def register_observed_frequency(self, attr_idx, observed_count, total):
        self.observed_freq[attr_idx] = observed_count / (total + self.epsilon)

    def compute_lambda(self, idx, modality, y_true_value=None):
        obs = self.observed_freq.get(idx, 0.1)
        λ_miss = (1.0 / (obs + self.epsilon)) ** 0.5
        λ_miss = float(np.clip(λ_miss, 0.75, 2.5))

        λ_cls = 1.0
        λ_num = 1.0

        if modality == "cat":
            if idx in self.class_counts and y_true_value is not None:
                counts_dict = self.class_counts[idx]
                count = counts_dict.get(int(y_true_value), 1)
                total_count = sum(counts_dict.values())
                num_classes = max(len(counts_dict), 1)

                avg_count = total_count / num_classes

                # Always > 1, larger for rarer classes
                λ_cls = 1.0 + (avg_count / (count + self.epsilon)) ** 0.5
                λ_cls = float(np.clip(λ_cls, 1.0, 3.0))

#                print(modality)
#                print(λ_cls)

            indicator_cat = 1.0
            indicator_num = 0.0

        elif modality == "num":
            #if idx in self.kde_dict and y_true_value is not None:
           #     density = self.kde_dict[idx](np.array([y_true_value], dtype=np.float64))[0]
           #     ref_density = self.kde_ref.get(idx, 1.0)

            #    λ_num = (ref_density / (density + self.epsilon)) ** 0.5
            #    λ_num = float(np.clip(λ_num, 1, 2.5))

              #  print(modality)
              #  print(λ_num)

            indicator_cat = 0.0
            indicator_num = 1.0

        elif modality == "txt":
            return λ_miss

        else:
            return None

        λ_i = λ_miss * (indicator_cat * λ_cls + indicator_num * λ_num)
        λ_i = float(np.clip(λ_i, 1, 3.0))
        return λ_i

    def forward(
        self,
        outputs,
        targets,
        modality_types,
        hidden_reps,
        retrieved_memory,
        attr_indices,
        modality_out,
        index_non_missing
    ):
        total_loss = 0.0
        valid_count = 0

        for i in range(len(modality_out)):
            modality = modality_out[i]
            y_pred = outputs[i]
            y_true = targets[i]
            idx = index_non_missing[i]

            if modality == "num":
                if y_true is None:
                    continue

                if not isinstance(y_true, torch.Tensor):
                    y_true = torch.tensor(y_true, dtype=y_pred.dtype, device=y_pred.device)
                else:
                    y_true = y_true.to(y_pred.device, dtype=y_pred.dtype)

                if torch.isnan(y_true).any() or torch.isnan(y_pred).any():
                    print("⚠️ Detected NaN in y_true or y_pred. Assigning zeros to both.")
                    y_true = torch.zeros_like(y_true)
                    y_pred = torch.zeros_like(y_pred)

                if y_pred.dim() == 0:
                    y_pred = y_pred.view(1, 1)
                elif y_pred.dim() == 1:
                    y_pred = y_pred.view(-1, 1)
                else:
                    y_pred = y_pred.view(y_pred.size(0), -1)

                if y_true.dim() == 0:
                    y_true = y_true.view(1, 1)
                elif y_true.dim() == 1:
                    y_true = y_true.view(-1, 1)
                else:
                    y_true = y_true.view(y_true.size(0), -1)

                if y_true.size(0) != y_pred.size(0):
                    if y_true.size(0) == 1:
                        y_true = y_true.expand(y_pred.size(0), y_true.size(1))
                    elif y_pred.size(0) == 1:
                        y_pred = y_pred.expand(y_true.size(0), y_pred.size(1))
                    else:
                        continue

                rec_loss = F.mse_loss(y_pred, y_true, reduction="none").mean(dim=1)

                λ_list = []
                for b in range(y_true.size(0)):
                    y_scalar = y_true[b, 0].item()
                    λ_i = self.compute_lambda(idx, modality, y_scalar)
                    λ_list.append(λ_i)

                λ_tensor = torch.tensor(λ_list, dtype=rec_loss.dtype, device=rec_loss.device)
                total_loss += self.num_scale * (λ_tensor * rec_loss).mean()
                valid_count += 1

            elif modality == "cat":
                if y_true is None:
                    continue

                if not isinstance(y_true, torch.Tensor):
                    y_true = torch.tensor(y_true, device=y_pred.device)
                else:
                    y_true = y_true.to(y_pred.device)

                y_true = y_true.long().view(-1)

                if y_pred.dim() == 1:
                    y_pred = y_pred.view(1, -1)
                elif y_pred.dim() > 2:
                    y_pred = y_pred.view(y_pred.size(0), -1)

                num_classes = y_pred.shape[1]

                if y_pred.size(0) != y_true.size(0):
                    if y_true.size(0) == 1 and y_pred.size(0) >= 1:
                        y_true = y_true.expand(y_pred.size(0))
                    elif y_pred.size(0) == 1 and y_true.size(0) >= 1:
                        y_pred = y_pred.expand(y_true.size(0), y_pred.size(1))
                    else:
                        continue

                valid_mask = (y_true >= 0) & (y_true < num_classes)
                if valid_mask.sum() == 0:
                    continue

                y_pred_valid = y_pred[valid_mask]
                y_true_valid = y_true[valid_mask]

                rec_loss = F.cross_entropy(y_pred_valid, y_true_valid, reduction="none")

                λ_list = []
                for b in range(y_true_valid.size(0)):
                    y_scalar = y_true_valid[b].item()
                    λ_i = self.compute_lambda(idx, modality, y_scalar)
                    λ_list.append(λ_i)

                λ_tensor = torch.tensor(λ_list, dtype=rec_loss.dtype, device=rec_loss.device)
                total_loss += self.cat_scale * (λ_tensor * rec_loss).mean()
                valid_count += 1

            elif modality == "txt":
                if y_true is None:
                    continue

                λ_i = self.compute_lambda(idx, modality)
                if λ_i is None:
                    continue

                if not isinstance(y_true, torch.Tensor):
                    y_true = torch.tensor(y_true, device=y_pred.device)
                else:
                    y_true = y_true.to(y_pred.device)

                if y_pred.dim() != 3:
                    continue

                y_pred = y_pred.permute(0, 2, 1)

                if y_true.dim() == 1:
                    y_true = y_true.unsqueeze(0)

                rec_loss = F.cross_entropy(y_pred, y_true.long(), reduction="mean")

                total_loss += self.txt_scale * λ_i * rec_loss
                valid_count += 1

            else:
                continue

        if valid_count == 0:
            device = hidden_reps[0].device if len(hidden_reps) > 0 else outputs[0].device
            return torch.tensor(0.0, requires_grad=True, device=device)

        return total_loss #/ valid_count

In [ ]:
import torch
import torch.nn.functional as F
from collections import Counter


def initialize_missing_embeddings(
    missing_attr_idx,
    known_attr,
    known_embedding,
    missing_mask,
    memory_bank,
    modality_types,
    k=5,
    r=2,
    d_h=384,
):
    """
    Initialize missing attribute embedding and optionally value using retrieved memory.

    This version follows the updated memory format:
      - txt modality: stores only embeddings
      - num/cat modality: stores embeddings and values

    Retrieval is based on Top-K similarity from observed attributes.
    Candidate memory indices are those that appear in at least r observed attributes.

    Args:
        missing_attr_idx (int): index of the missing attribute to initialize
        known_attr (List[int]): indices of observed attributes
        known_embedding (Tensor): shape [num_attributes, d_h]
        missing_mask (List[bool] or Tensor): unused here except for compatibility
        memory_bank (PerAttributeMemoryBank): updated memory bank
        modality_types (List[str]): modality per attribute ('txt', 'num', 'cat')
        k (int): Top-K neighbors per observed attribute
        r (int): minimum number of observed attributes that must agree on a candidate
        d_h (int): embedding size

    Returns:
        init_embedding (Tensor): shape [d_h]
        init_value: inferred value for num/cat, None for txt
        retrieved_indices (List[int]): selected candidate memory indices
    """
    device = known_embedding.device
    missing_modality = modality_types[missing_attr_idx]

    # ---------------------------------------------------------
    # Step 1: retrieve Top-K memory indices for each observed attribute
    # ---------------------------------------------------------
    topk_lists = []

    for i in known_attr:
        mem = memory_bank.get(i, return_values=False)
        if mem is None or mem.size(0) == 0:
            continue

        mem = mem.to(device)

        # remove zero vectors
        non_zero_mask = mem.abs().sum(dim=1) > 1e-6
        mem = mem[non_zero_mask]

        if mem.size(0) == 0:
            continue

        # cosine similarity retrieval
        query = F.normalize(known_embedding[i], dim=0)                # [d_h]
        mem_norm = F.normalize(mem, dim=1)                            # [N, d_h]
        sim = torch.matmul(mem_norm, query)                           # [N]

        topk = torch.topk(sim, k=min(k, mem.size(0))).indices.tolist()
        topk_lists.append(topk)

    # ---------------------------------------------------------
    # Step 2: find candidate indices appearing in at least r lists
    # ---------------------------------------------------------
    candidate_indices = []
    if len(topk_lists) > 0:
        counter = Counter()
        for lst in topk_lists:
            counter.update(lst)

        candidate_indices = [idx for idx, cnt in counter.items() if cnt >= r]

        # fallback: if no index appears at least r times, use the most frequent ones
        if len(candidate_indices) == 0:
            if len(counter) > 0:
                max_count = max(counter.values())
                candidate_indices = [idx for idx, cnt in counter.items() if cnt == max_count]

    # ---------------------------------------------------------
    # Step 3: retrieve memory of the missing attribute
    # ---------------------------------------------------------
    mem_missing, values_missing = memory_bank.get(missing_attr_idx, return_values=True)

    if mem_missing is None or mem_missing.size(0) == 0:
        return torch.zeros(d_h, device=device), None, []

    mem_missing = mem_missing.to(device)

    # remove zero vectors and keep aligned values
    non_zero_mask = mem_missing.abs().sum(dim=1) > 1e-6
    mem_missing = mem_missing[non_zero_mask]
    values_missing = [v for j, v in enumerate(values_missing) if non_zero_mask[j].item()]

    if mem_missing.size(0) == 0:
        return torch.zeros(d_h, device=device), None, []

    # keep only valid candidate indices
    candidate_indices = [idx for idx in candidate_indices if idx < mem_missing.size(0)]

    # fallback: use all memory if candidate set is empty
    if len(candidate_indices) == 0:
        selected_embeddings = mem_missing
        selected_values = values_missing
        retrieved_indices = list(range(mem_missing.size(0)))
    else:
        selected_embeddings = mem_missing[candidate_indices]
        selected_values = [values_missing[idx] for idx in candidate_indices]
        retrieved_indices = candidate_indices

    # ---------------------------------------------------------
    # Step 4: initialize embedding
    # ---------------------------------------------------------
    init_embedding = selected_embeddings.mean(dim=0)

    # ---------------------------------------------------------
    # Step 5: initialize value depending on modality
    # ---------------------------------------------------------
    init_value = None

    if missing_modality == "num":
        valid_values = []
        for v in selected_values:
            if v is None:
                continue
            if isinstance(v, torch.Tensor):
                if v.numel() == 1:
                    valid_values.append(v.item())
            else:
                try:
                    valid_values.append(float(v))
                except (TypeError, ValueError):
                    pass

        if len(valid_values) > 0:
            init_value = sum(valid_values) / len(valid_values)

    elif missing_modality == "cat":
        valid_values = [v for v in selected_values if v is not None]
        if len(valid_values) > 0:
            init_value = Counter(valid_values).most_common(1)[0][0]

    elif missing_modality == "txt":
        init_value = None

    return init_embedding, init_value, retrieved_indices

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
vocab_size = tokenizer.vocab_size


class RAGNet(nn.Module):
    def __init__(self,
                 d_llm=384,
                 d_num=32,
                 d_cat=16,
                 d_shared=128,
                 num_numerical_attrs=14,
                 num_categorical_attrs=20,
                 num_classes_per_cat_attr=None,
                 vocab_size=10000,
                 decoder_dim=384,
                 max_text_len=64,
                 loss_fn=None,
                 PerAttributeMemoryBank1=None):
        super().__init__()

        self.projection = RAGNetProjectionModule(
            d_llm=d_llm, d_num=d_num, d_cat=d_cat, d_shared=d_shared,
            num_numerical_attrs=num_numerical_attrs,
            num_categorical_attrs=num_categorical_attrs,
            num_classes_per_cat_attr=num_classes_per_cat_attr
        )

        self.total_attrs = num_numerical_attrs + num_categorical_attrs

        self.gnn = CrossModalGNNLayer(d_shared)

        self.memory_bank = PerAttributeMemoryBank1
        self.memory_fusion = AttributeMemoryFusion(d_h=384, memory_bank=self.memory_bank)

        self.num_heads = nn.ModuleList([NumericalHead(d_shared) for _ in range(num_numerical_attrs)])
        self.cat_heads = nn.ModuleList([
            CategoricalHead(d_shared, num_cls) for num_cls in num_classes_per_cat_attr
        ])
        self.txt_decoder = TextualDecoder(d_shared, vocab_size, max_len=max_text_len, decoder_dim=decoder_dim)

        self.loss_module = loss_fn

    def forward(self, llm_embeddings, raw_numericals, cat_indices, missing_mask, edge_index,
                modality_types, targets=None, attr_indices=None, mode=None):

        device = next(self.parameters()).device

        llm_embeddings = llm_embeddings.to(device)
        edge_index = edge_index.to(device)

        if isinstance(raw_numericals, torch.Tensor):
            raw_numericals = raw_numericals.to(device)
        if isinstance(cat_indices, torch.Tensor):
            cat_indices = cat_indices.to(device)
        if isinstance(missing_mask, torch.Tensor):
            missing_mask = missing_mask.to(device)

        Z = []
        out_preds = []
        retrieved_mem = []
        final_reps = []
        used_targets = []
        modality_out = []
        used_mods = []
        used_indices = []

        numerical_idx = 0
        num_attr_idx = 0
        cat_attr_idx = 0
        txt_attr_idx = 0

        # --------------------------------------------------
        # Step 1: detect known / missing attributes from LLM embeddings
        # --------------------------------------------------
        non_zero_mask = llm_embeddings.abs().sum(dim=1) > 1e-6
        known_attr = torch.where(non_zero_mask)[0].tolist()
        missing_attr = torch.where(~non_zero_mask)[0].tolist()

        # Make local copies because we may fill missing values from memory
        if isinstance(raw_numericals, torch.Tensor):
            raw_numericals_filled = raw_numericals.clone().to(device)
        else:
            raw_numericals_filled = list(raw_numericals)

        if isinstance(cat_indices, torch.Tensor):
            cat_indices_filled = cat_indices.clone().to(device)
        else:
            cat_indices_filled = list(cat_indices)

        retrieved_init_info = {}

        # --------------------------------------------------
        # Step 2: initialize missing embeddings and values from memory
        # --------------------------------------------------
        for m_attr in missing_attr:
            init_embedding, init_value, retrieved_indices = initialize_missing_embeddings(
                missing_attr_idx=m_attr,
                known_attr=known_attr,
                known_embedding=llm_embeddings,
                missing_mask=(~non_zero_mask).tolist(),
                memory_bank=self.memory_bank,
                modality_types=modality_types,
                k=5,
                r=2,
                d_h=llm_embeddings.size(1),
            )

            if init_embedding is not None and isinstance(init_embedding, torch.Tensor):
                llm_embeddings[m_attr] = init_embedding.to(device)

            retrieved_init_info[m_attr] = {
                "value": init_value,
                "indices": retrieved_indices
            }

        # --------------------------------------------------
        # Step 3: fill missing raw num/cat values using retrieved memory values
        # --------------------------------------------------
        numerical_counter = 0
        categorical_counter = 0

        for i, modality in enumerate(modality_types):
            if modality == "num":
                if i in missing_attr and i in retrieved_init_info:
                    init_value = retrieved_init_info[i]["value"]
                    if init_value is not None:
                        if isinstance(raw_numericals_filled, torch.Tensor):
                            raw_numericals_filled[numerical_counter] = torch.tensor(
                                init_value,
                                dtype=raw_numericals_filled.dtype,
                                device=device
                            )
                        else:
                            raw_numericals_filled[numerical_counter] = float(init_value)
                numerical_counter += 1

            elif modality == "cat":
                if i in missing_attr and i in retrieved_init_info:
                    init_value = retrieved_init_info[i]["value"]
                    if init_value is not None:
                        if isinstance(cat_indices_filled, torch.Tensor):
                            cat_indices_filled[categorical_counter] = torch.tensor(
                                int(init_value),
                                dtype=cat_indices_filled.dtype,
                                device=device
                            )
                        else:
                            cat_indices_filled[categorical_counter] = int(init_value)
                categorical_counter += 1

        # --------------------------------------------------
        # Step 4: projection
        # --------------------------------------------------
        numerical_idx = 0
        cat_attr_idx = 0

        for i, modality in enumerate(modality_types):
            if modality == 'num':
                raw_num_val = raw_numericals_filled[numerical_idx]
                if not isinstance(raw_num_val, torch.Tensor):
                    raw_num_val = torch.tensor(raw_num_val, dtype=torch.float32, device=device)
                else:
                    raw_num_val = raw_num_val.to(device=device, dtype=torch.float32)

                emb = self.projection.forward_numerical(
                    numerical_idx,
                    llm_embeddings[i].to(device),
                    raw_num_val
                )
                numerical_idx += 1

            elif modality == 'cat':
                cat_class_val = cat_indices_filled[cat_attr_idx]

                if isinstance(cat_class_val, torch.Tensor):
                    cat_class_tensor = cat_class_val.to(device).long().view(1)
                else:
                    cat_class_tensor = torch.tensor(
                        [int(cat_class_val)],
                        dtype=torch.long,
                        device=device
                    )

                emb = self.projection.forward_categorical(
                    cat_attr_idx,
                    llm_embeddings[i].unsqueeze(0).to(device),
                    cat_class_tensor
                ).squeeze(0)
                cat_attr_idx += 1

            elif modality == 'txt':
                emb = llm_embeddings[i].to(device)

            else:
                raise ValueError(f"Unknown modality: {modality}")

            Z.append(emb.to(device))

        Z = torch.stack(Z, dim=0)
        Z_gnn = self.gnn(Z, edge_index)
        final_reps = Z_gnn

        # --------------------------------------------------
        # Step 5: memory fusion
        # --------------------------------------------------
        Z_mem = self.memory_fusion(Z_gnn)

        diff_vectors = Z_mem - Z_gnn
        mask = ~torch.isnan(diff_vectors).any(dim=1)

        diff_l2 = torch.norm(diff_vectors[mask], dim=1)
        Z_mem_masked = Z_mem[mask]
        Z_gnn_masked = Z_gnn[mask]
        diff_cos = F.cosine_similarity(Z_mem_masked, Z_gnn_masked, dim=1)

        self.gnn_mem_diff_l2 = diff_l2
        self.gnn_mem_diff_cos = diff_cos

        numerical_idx = 0
        num_attr_idx = 0
        cat_attr_idx = 0
        txt_attr_idx = 0
        index_non_missing = []

        # --------------------------------------------------
        # Step 6: prediction heads
        # --------------------------------------------------
        for i, modality in enumerate(modality_types):
            index_non_missing.append(i)

            if modality == 'num':
                if i in missing_attr and mode == 'train':
                    out_preds.append(0 * self.num_heads[num_attr_idx](Z_mem[i].unsqueeze(0)))
                    y_true = targets[i]
                    if not isinstance(y_true, torch.Tensor):
                        y_true = torch.tensor(y_true, dtype=torch.float32, device=device)
                    else:
                        y_true = y_true.to(device=device, dtype=torch.float32)
                    if torch.isnan(y_true):
                        y_true = torch.tensor(0.0, dtype=torch.float32, device=device)
                    used_targets.append(0 * y_true)
                    modality_out.append(modality)
                else:
                    out_preds.append(self.num_heads[num_attr_idx](Z_mem[i].unsqueeze(0)))
                    y_true = targets[i]
                    if not isinstance(y_true, torch.Tensor):
                        y_true = torch.tensor(y_true, dtype=torch.float32, device=device)
                    else:
                        y_true = y_true.to(device=device, dtype=torch.float32)
                    used_targets.append(y_true)
                    modality_out.append(modality)

                num_attr_idx += 1

            elif modality == 'cat':
                if i in missing_attr and mode == 'train':
                    out_preds.append(0 * self.cat_heads[cat_attr_idx](Z_mem[i].unsqueeze(0)))
                    used_targets.append(0 * torch.tensor([targets[i]], device=device))
                    modality_out.append(modality)
                else:
                    out_preds.append(self.cat_heads[cat_attr_idx](Z_mem[i].unsqueeze(0)))
                    used_targets.append(torch.tensor([targets[i]], device=device))
                    modality_out.append(modality)

                cat_attr_idx += 1

            elif modality == 'txt':
                tokenized = tokenizer(
                    targets[i],
                    return_tensors="pt",
                    padding='max_length',
                    truncation=True,
                    max_length=32
                ).input_ids.to(device)

                out = self.txt_decoder(
                    Z_mem[i].unsqueeze(0),
                    target=tokenized,
                    teacher_forcing=True
                )

                if i in missing_attr and mode == 'train':
                    out_preds.append(0 * out)
                    used_targets.append(0 * tokenized)
                    modality_out.append(modality)
                else:
                    out_preds.append(out)
                    used_targets.append(tokenized)
                    modality_out.append(modality)

                txt_attr_idx += 1

        loss = self.loss_module(
            out_preds,
            used_targets,
            modality_types,
            final_reps,
            retrieved_mem,
            used_indices,
            modality_out,
            index_non_missing
        )

        loss = loss + (diff_l2.mean() / 38)
        return loss, out_preds

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
import pandas as pd

# Example: load your dataset
df = pd.read_csv("all_data_after2_updated.csv", dtype=str)

# Define categorical columns (replace with your actual list)
#categorical_columns = [
#    'Classification of the physical effects',
#    'Nature of the consequences',
#    'Release type',
    # add more if needed
#]
#categorical_indices = [1, 2, 3, 8, 21, 22, 23,  27, 29, 31, 35, 36, 37]
categorical_indices= [1, 2, 3, 8, 21, 22, 23,  27, 29, 31, 35, 36, 37]

# Exactly 38 total attributes
#categorical_indices = [1, 2, 3, 4, 8, 9, 15, 19, 21, 22, 23, 24, 25, 27, 29, 31, 35, 36, 37, 41]
categorical_columns = [df.columns[i] for i in categorical_indices]

# Compute number of unique non-null categories for each attribute
num_classes_per_cat_attr = [df[col].dropna().nunique() for col in categorical_columns]

print("Number of classes per categorical attribute:", num_classes_per_cat_attr)


Number of classes per categorical attribute: [2, 3, 5, 4, 2, 2, 2, 12, 12, 4, 4, 16, 2]


In [ ]:
model=RAGNet(d_llm=384,d_num=32,d_cat=16,d_shared=384,num_numerical_attrs=15,num_categorical_attrs=20,
                 num_classes_per_cat_attr=num_classes_per_cat_attr,
                 vocab_size=10000,
                 decoder_dim=384,
                 max_text_len=64)

In [ ]:
import numpy as np

# Load embeddings
textual_embeddings = np.load("textual_embeddings.npy", allow_pickle=True)
categorical_embeddings = np.load("categorical_embeddings_reduced.npy", allow_pickle=True)
numerical_embeddings = np.load("numerical_embeddings_reduced.npy", allow_pickle=True)

# Show dimensions
print("Textual Embeddings Shape:", textual_embeddings.shape)
print("Categorical Embeddings Shape (original):", categorical_embeddings.shape)
print("Categorical Embeddings Shape (reduced):", categorical_embeddings.shape)
print("Numerical Embeddings Shape:", numerical_embeddings.shape)

# Concatenate along the feature dimension (axis=1)
all_embeddings_np = np.concatenate(
    [textual_embeddings, categorical_embeddings, numerical_embeddings], axis=1
)

# Final shape
print("Combined Embeddings Shape (all_embeddings_np):", all_embeddings_np.shape)

# Save the new combined embeddings
np.save("all_embeddings_reduced.npy", all_embeddings_np)

# (Optional) Save the reduced categorical embeddings separately


Textual Embeddings Shape: (755, 3, 384)
Categorical Embeddings Shape (original): (755, 13, 384)
Categorical Embeddings Shape (reduced): (755, 13, 384)
Numerical Embeddings Shape: (755, 9, 384)
Combined Embeddings Shape (all_embeddings_np): (755, 25, 384)


In [ ]:
import torch

# all_embeddings should be a 3D tensor: [N, M, D]
# where:
#   N = number of samples
#   M = number of attributes (e.g., 3 modalities)
#   D = embedding dimension (e.g., 384)
all_embeddings = torch.from_numpy(all_embeddings_np)
N, M, D = all_embeddings.shape

# Initialize tensor to hold normalized embeddings
normalized_embeddings = all_embeddings.clone()

for m in range(M):  # For each attribute/modality
    attr_embeddings = all_embeddings[:, m, :]  # Shape: [N, D]

    # Mask: find non-zero vectors
    non_zero_mask = (attr_embeddings.abs().sum(dim=1) != 0)

    # Compute mean and std for non-zero rows only
    means = attr_embeddings[non_zero_mask].mean(dim=0)
    stds = attr_embeddings[non_zero_mask].std(dim=0)

    # Normalize only non-zero vectors
    normalized_embeddings[:, m, :][non_zero_mask] = (
        attr_embeddings[non_zero_mask] - means
    ) / (stds + 1e-8)

print("✅ Normalization complete. Shape:", normalized_embeddings.shape)


✅ Normalization complete. Shape: torch.Size([755, 25, 384])


In [ ]:
!pip install python-Levenshtein


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 57.6 MB/s eta 0:00:00


In [ ]:
from sklearn.model_selection import train_test_split
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score

# ----------- Split Data -----------
def split_train_test(normalized_embeddings, ground_truth, test_ratio=0.2):
    indices = np.arange(len(normalized_embeddings))
    train_idx, test_idx = train_test_split(indices, test_size=test_ratio, random_state=42)
    return train_idx, test_idx
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
id2word = tokenizer.convert_ids_to_tokens
import torch
import random
import torch.nn.functional as F
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from Levenshtein import distance as levenshtein
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
# ----------- Evaluate -----------
def evaluate_per_sample(model, normalized_embeddings, ground_truth, test_idx, modalities, cat_maps, loss_fn,edge_index, modality_types, numerical_values, categorical_indices):
    model.eval()
    results = defaultdict(list)
    total_eval_loss = 0.0
    total_valid_count = 0

    with torch.no_grad():
      for idx in test_idx:
        for modality in ['num', 'cat', 'txt']:
            emb = normalized_embeddings[idx]
            gt = ground_truth[idx]
            missing_mask = (gt == -1)
          # Step 1: Find non-zero rows
            non_zero_mask = emb.abs().sum(dim=1) > 1e-6
            non_zero_indices = non_zero_mask.nonzero(as_tuple=True)[0].tolist()
# Step 2: Group indices by modality
            modality_indices = defaultdict(list)
            modality_indices={'txt': [0, 1, 2], 'cat': [3, 4, 5, 6, 7, 8, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22], 'num': [23, 26, 27, 31, 34, 35, 36, 37]}
            mapped_modality_indices =[]# defaultdict(list)
# Step 4: Iterate through non-zero indices and map them to their corresponding modality
#            indices=2#
            for idx in non_zero_indices:
                for indices in modality_indices[modality]:
                     if (idx==indices):
                         mapped_modality_indices.append(idx)
            if len(mapped_modality_indices):
              masked_attr_idx = random.choice(mapped_modality_indices)
              print(masked_attr_idx)
           #   print(idx)
             # print(mapped_modality_indices)

            else:
             # print(f"Warning: No indices for modality {modality}. Skipping this modality.")
              continue  # Skip to the next iteration if the modality is empty

#            masked_attr_idx = random.choice(mapped_modality_indices[modality])
            #print(masked_attr_idx)
            emb[masked_attr_idx] = 0.0
            missing_mask[masked_attr_idx] = True
            loss, outputs = model(
            llm_embeddings=emb,
            raw_numericals=numerical_values[idx],
            cat_indices=categorical_indices[idx],
            missing_mask=missing_mask,
            edge_index=edge_index,
            modality_types=modalities,
            targets=ground_truth[idx],
            attr_indices=list(range(len(modalities))),mode='test')
            total_eval_loss += loss.item()
            smooth = SmoothingFunction().method4  # Recommended for short sentences
           # for i in range(min(len(gt), len(outputs))):
            for i, mod in enumerate(modalities):
            #  print(i)

              if masked_attr_idx is not None:
               if i==masked_attr_idx:
                gt_i = gt[i]
                out_i = outputs[i]
                if mod == 'num':
                    results['num_mse'].append((gt_i - out_i)**2)
                    results['num_mae'].append(abs(gt_i - out_i))
                elif mod == 'cat':
                  gt_val = gt_i.item() if hasattr(gt_i, 'item') else gt_i
                  pred_val = out_i.argmax().item()
                  results['cat_acc'].append(int(gt_val == pred_val))
                  results['cat_f1'].append(f1_score([gt_val], [pred_val], average='macro'))
                elif mod == 'txt':
                  if isinstance(gt_i, torch.Tensor):
                    gt_tokens = gt_i.view(-1).tolist()
                  elif isinstance(gt_i, str):
                    gt_tokens = tokenizer.encode(gt_i, add_special_tokens=False)
                  elif isinstance(gt_i, (list, tuple)):
                    gt_tokens = list(gt_i)
                  elif isinstance(gt_i, int):
                    gt_tokens = [gt_i]
                  else:
                    raise TypeError(f"Unsupported type for gt_i: {type(gt_i)}")
                  if out_i.dim() == 3:
                    pred_ids = out_i.argmax(dim=-1).squeeze(0)  # (T,)
                  elif out_i.dim() == 2:
                    pred_ids = out_i.argmax(dim=-1)  # (T,)
                  else:
                    raise ValueError(f"Unexpected output shape for logits: {out_i.shape}")
                  ignore_tokens = {0, 101, 102}  # PAD, [CLS], [SEP]
                  gt_ids = [int(t) for t in gt_tokens if isinstance(t, int) and t not in ignore_tokens]
                  out_ids = [int(t.item()) for t in pred_ids if t.item() not in ignore_tokens]
                  # ---- Token-level Accuracy ----
                  min_len = min(len(gt_ids), len(out_ids))
                  if min_len == 0:
                    results['txt_token_acc'].append(0.0)
                    continue

                  correct = sum(g == o for g, o in zip(gt_ids[:min_len], out_ids[:min_len]))
                  accuracy = correct / min_len
                  results['txt_token_acc'].append(accuracy)
                  # Ignore special tokens
                  ignore_tokens = {0, 101, 102}  # PAD, [CLS], [SEP]

                  gt_ids = [t for t in gt_ids if t not in ignore_tokens]
                  out_ids = [t for t in out_ids if t not in ignore_tokens]

# Convert token IDs to string (words or subwords)
                  ref_sentence = tokenizer.decode(gt_ids, skip_special_tokens=True)
                  hyp_sentence = tokenizer.decode(out_ids, skip_special_tokens=True)

# Convert to word lists
                  ref_words = ref_sentence.split()
                  hyp_words = hyp_sentence.split()

# Compute BLEU (optional smoothing for short sentences)

                  smooth = SmoothingFunction().method4

                  if len(ref_words) > 0 and len(hyp_words) > 0:
                    bleu_score = sentence_bleu([ref_words], hyp_words, smoothing_function=smooth)
                  else:
                    bleu_score = 0.0
                  results['bleu_score'].append(bleu_score)
                total_valid_count += 1

    results = {k: np.mean(v) for k, v in results.items() if v}
    results['total_loss'] = total_eval_loss / max(total_valid_count, 1)
    print(results)
    return results

# ----------- Train & Evaluate -----------
def train_and_evaluate(
    model,
    normalized_embeddings,
    ground_truth,
    numerical_values,
    categorical_indices,
    modalities,
    cat_maps,
    loss_fn,
    edge_index,
    device,
    epochs=10,
    test_ratio=0.2
):


    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    train_idx, test_idx = split_train_test(normalized_embeddings, ground_truth, test_ratio)
    #epochs=10
    idx1=0
    for epoch in range(100):
        model.train()
        total_loss = 0.0
        optimizer = torch.optim.Adam(model.parameters(), lr=1e-5)
        train_idx, test_idx = split_train_test(normalized_embeddings, ground_truth, test_ratio)
#        metrics = evaluate_per_sample(model, normalized_embeddings, ground_truth, test_idx, modalities, cat_maps, loss_fn,edge_index=edge_index,modality_types=modalities,numerical_values=numerical_values,categorical_indices=categorical_indices)


        for idx in train_idx:
            optimizer.zero_grad()
 #           print(idx1)
            idx1=idx1+1
            emb = normalized_embeddings[idx]
            gt = ground_truth[idx]
            missing_mask = (gt == -1)
            loss, _ = model(
            llm_embeddings=emb,
            raw_numericals=numerical_values[idx],
            cat_indices=categorical_indices[idx],
            missing_mask=missing_mask,
            edge_index=edge_index,
            modality_types=modalities,
            targets=ground_truth[idx],
            attr_indices=list(range(len(modalities))))


           # loss, _ = model(emb, gt, missing_mask)
        #    optimizer.zero_grad()

            if not torch.isnan(loss):
              loss.backward()
              optimizer.step()
              #print(loss)

            total_loss += loss.item()


        print(f"[Epoch {epoch+1}/{epochs}] Avg Train Loss: {total_loss / len(train_idx):.4f}")
        if epoch % 5 ==0:
          print("🔍 Running Evaluation...")
          metrics = evaluate_per_sample(model, normalized_embeddings, ground_truth, test_idx, modalities, cat_maps, loss_fn,edge_index=edge_index,modality_types=modalities,numerical_values=numerical_values,categorical_indices=categorical_indices)
          print(metrics)
                # Save model after epoch 20
        if epoch + 1 == 20:
            torch.save(model.state_dict(), "model_epoch20.pt")
            print("💾 Model saved at epoch 20!")


 #   metrics = evaluate_per_sample(model, normalized_embeddings, ground_truth, test_idx, modalities, cat_maps, loss_fn,missing_mask=missing_mask,edge_index=edge_index,modality_types=modalities)
    print("📊 Evaluation Results:", metrics)
    return metrics


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
!pip install nltk


In [ ]:

import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")  # or your custom tokenizer

# Load data
df = pd.read_csv("all_data_after2_updated.csv", dtype=str)

# Define attribute columns
textual_columns = ['Event full description', 'Lesson Learnt', 'Post-event summary']
categorical_indices= [1, 2, 3, 8, 21, 22, 23,  27, 29, 31, 35, 36, 37]

#categorical_indices = [1, 2, 3, 4, 8, 9, 15, 19, 21, 22, 23, 24, 25, 27, 29, 31, 35, 36, 37, 41]
#numerical_indices = [10, 11, 12, 13, 14, 16, 17,18, 20, 26, 32, 33, 34, 39, 40]
numerical_indices = [10,  13, 14, 20,  32, 33, 34, 39, 40]
categorical_columns = [df.columns[i] for i in categorical_indices]
numerical_columns = [df.columns[i] for i in numerical_indices]

# Initialize ground truth lists
ground_truth_text = []
ground_truth_cat = []
ground_truth_num = []

# ----------- 1. Textual Columns -----------
for col in textual_columns:
    ground_truth_text.append(df[col].fillna(""))

# ----------- 2. Categorical Columns -----------
cat_maps = {}
for col in categorical_columns:
    le = LabelEncoder()
    values = df[col].fillna("Missing")  # Treat missing as a class
    encoded = le.fit_transform(values)
    ground_truth_cat.append(encoded)
    cat_maps[col] = le

# Stack categorical attributes
ground_truth_cat = np.stack(ground_truth_cat, axis=1)  # Shape: (N, num_cat)

# ----------- 3. Numerical Columns -----------
num_df = df[numerical_columns].apply(pd.to_numeric, errors='coerce')  # Convert to numeric
scaler = StandardScaler()
ground_truth_num = scaler.fit_transform(num_df.fillna(np.nan))

# Where values were missing, set to -1 for masking later
ground_truth_num[np.isnan(ground_truth_num)] = -1

# ----------- 4. Combine All Modalities -----------
ground_truth_text = np.array(ground_truth_text).T  # Shape: (N, num_txt)
final_ground_truth = np.concatenate([ground_truth_text, ground_truth_cat, ground_truth_num], axis=1)

print("✅ Final Ground Truth Shape:", final_ground_truth.shape)
ground_truth=final_ground_truth


✅ Final Ground Truth Shape: (755, 25)


In [ ]:
ground_truth=final_ground_truth

In [ ]:
import os
import json
import random
import numpy as np
import torch

# ============================================================
# TRAINING HELPERS
# ============================================================

def load_fixed_split(split_json_path=None, mask_file_path=None):
    """
    Load fixed train/test split from:
      - fixed_setup.json
      - or a mask file containing train_idx/test_idx
    """
    if mask_file_path is not None:
        with open(mask_file_path, "r") as f:
            payload = json.load(f)
        train_idx = np.array(payload["train_idx"], dtype=int)
        test_idx = np.array(payload["test_idx"], dtype=int)
        print(f"Loaded split from mask file: {mask_file_path}")
        return train_idx, test_idx

    if split_json_path is not None:
        with open(split_json_path, "r") as f:
            payload = json.load(f)
        train_idx = np.array(payload["train_idx"], dtype=int)
        test_idx = np.array(payload["test_idx"], dtype=int)
        print(f"Loaded split from fixed setup: {split_json_path}")
        return train_idx, test_idx

    raise ValueError("Provide either split_json_path or mask_file_path")


def build_missing_mask_from_gt(gt_row, device=None):
    mask = []
    for x in gt_row:
        if isinstance(x, str):
            s = x.strip()
            mask.append(s == "" or s == "-1" or s.lower() == "nan")
        else:
            try:
                mask.append(float(x) == -1)
            except Exception:
                mask.append(False)
    return torch.tensor(mask, dtype=torch.bool, device=device)


# ============================================================
# TRAIN ONLY WITH FIXED SPLIT
# ============================================================

def train_with_fixed_split(
    model,
    normalized_embeddings,
    ground_truth,
    numerical_values,
    categorical_indices,
    modalities,
    edge_index,
    device,
    split_json_path=None,
    mask_file_path=None,
    epochs=50,
    batch_size=16,
    lr=1e-4,
    save_model_path="trained_model_fixed_split.pt"
):
    """
    Train model using a fixed train/test split loaded from:
      - fixed_setup.json
      - or one uploaded mask file

    This function only trains and returns:
      model, train_idx, test_idx
    """
    train_idx, test_idx = load_fixed_split(
        split_json_path=split_json_path,
        mask_file_path=mask_file_path
    )

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    if isinstance(normalized_embeddings, np.ndarray):
        normalized_embeddings = torch.tensor(normalized_embeddings, dtype=torch.float32)

    normalized_embeddings = normalized_embeddings.to(device)

    if isinstance(numerical_values, np.ndarray):
        numerical_values = torch.tensor(numerical_values, dtype=torch.float32, device=device)
    else:
        numerical_values = numerical_values.to(device)

    if isinstance(categorical_indices, np.ndarray):
        categorical_indices = torch.tensor(categorical_indices, dtype=torch.long, device=device)
    else:
        categorical_indices = categorical_indices.to(device)

    print("\n===== TRAINING WITH FIXED SPLIT =====")
    print("Train size:", len(train_idx))
    print("Test size:", len(test_idx))

    for epoch in range(epochs):
        model.train()
        total_loss = 0.0

        random_indices = np.random.permutation(train_idx)

        for start in range(0, len(random_indices), batch_size):
            batch_indices = random_indices[start:start + batch_size]
            optimizer.zero_grad()

            batch_loss = None

            for idx in batch_indices:
                emb = normalized_embeddings[idx]
                gt = ground_truth[idx]
                missing_mask = build_missing_mask_from_gt(gt, emb.device)

                loss, _ = model(
                    llm_embeddings=emb,
                    raw_numericals=numerical_values[idx],
                    cat_indices=categorical_indices[idx],
                    missing_mask=missing_mask,
                    edge_index=edge_index,
                    modality_types=modalities,
                    targets=gt,
                    attr_indices=list(range(len(modalities))),
                    mode='train'
                )

                if loss is not None and not torch.isnan(loss) and not torch.isinf(loss):
                    if batch_loss is None:
                        batch_loss = loss
                    else:
                        batch_loss = batch_loss + loss

            if batch_loss is not None and not torch.isnan(batch_loss) and not torch.isinf(batch_loss):
                batch_loss = batch_loss / len(batch_indices)
                batch_loss.backward()
                optimizer.step()
                total_loss += batch_loss.item()

        avg_train_loss = total_loss / max((len(train_idx) / batch_size), 1)
        print(f"[Epoch {epoch+1}/{epochs}] Avg Train Loss: {avg_train_loss:.4f}")

    torch.save(model.state_dict(), save_model_path)
    print(f"\nSaved trained model to: {save_model_path}")

    return model, train_idx, test_idx

In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd

from collections import defaultdict
from sklearn.preprocessing import LabelEncoder
from transformers import AutoTokenizer

# -------------------------------------------------
# Reproducibility
# -------------------------------------------------
torch.manual_seed(42)
np.random.seed(42)

# -------------------------------------------------
# Tokenizer
# -------------------------------------------------
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
vocab_size = tokenizer.vocab_size

# -------------------------------------------------
# Build modality list
# -------------------------------------------------
modalities = ['txt'] * len(textual_columns) + ['cat'] * len(categorical_columns) + ['num'] * len(numerical_columns)

# -------------------------------------------------
# Category encoders / maps
# -------------------------------------------------
cat_maps = {}
cat_label_encoders = {}

for idx, col in enumerate(categorical_columns):
    encoder = LabelEncoder()
    non_null_vals = df[col].fillna("MISSING").astype(str)
    encoder.fit(non_null_vals)
    cat_maps[idx] = encoder
    cat_label_encoders[col] = encoder

# number of classes per categorical attribute
num_classes_per_cat_attr = [len(cat_maps[i].classes_) for i in range(len(categorical_columns))]

# -------------------------------------------------
# Loss
# -------------------------------------------------
loss_fn = MARIPLoss(d_h=128)

# -------------------------------------------------
# Register priors for each attribute
# -------------------------------------------------
num_attrs = len(modalities)

for attr_idx in range(num_attrs):
    modality = modalities[attr_idx]
    column = ground_truth[:, attr_idx]

    try:
        if modality == "num":
            observed = column[column != -1]
            values = observed.astype(np.float32)
            if len(values) > 1:
                loss_fn.register_numerical_kde(attr_idx, values)
            loss_fn.register_observed_frequency(attr_idx, len(observed), len(column))

        elif modality == "cat":
            observed = column[column != -1]
            class_labels = observed.astype(int)
            if len(class_labels) > 0:
                loss_fn.register_class_counts(attr_idx, class_labels)
            loss_fn.register_observed_frequency(attr_idx, len(observed), len(column))

        elif modality == "txt":
            observed_texts = [str(v) for v in column if str(v).strip() not in ("", "-1", "nan", "None")]
            loss_fn.register_observed_frequency(attr_idx, len(observed_texts), len(column))
            text_lengths = [len(text.split()) for text in observed_texts]
            avg_length = np.mean(text_lengths) if text_lengths else 0
            print(f"📄 Text Attr {attr_idx}: {len(observed_texts)} observed, Avg. Length: {avg_length:.2f}")

    except Exception as e:
        print(f"⚠️ Registration failed for attr {attr_idx}: {e}")

# -------------------------------------------------
# Load embeddings
# -------------------------------------------------
textual_embeddings = np.load("textual_embeddings.npy", allow_pickle=True)
categorical_embeddings = np.load("categorical_embeddings_reduced.npy", allow_pickle=True)
numerical_embeddings = np.load("numerical_embeddings_reduced.npy", allow_pickle=True)

textual_embeddings = torch.tensor(textual_embeddings, dtype=torch.float32)
categorical_embeddings = torch.tensor(categorical_embeddings, dtype=torch.float32)
numerical_embeddings = torch.tensor(numerical_embeddings, dtype=torch.float32)

all_embeddings = torch.cat(
    [textual_embeddings, categorical_embeddings, numerical_embeddings],
    dim=1
)  # (N, M, D)

normalized_embeddings = all_embeddings  # if already normalized; otherwise replace with your normalized tensor

# -------------------------------------------------
# Raw values for num / cat
# -------------------------------------------------
numerical_values = df[numerical_columns].apply(pd.to_numeric, errors='coerce').fillna(-1).to_numpy()

categorical_indices_df = pd.DataFrame()
for col in categorical_columns:
    encoder = cat_label_encoders[col]
    vals = df[col].fillna("MISSING").astype(str)
    categorical_indices_df[col] = encoder.transform(vals)

categorical_indices = categorical_indices_df.to_numpy()
textual_values = df[textual_columns].fillna("").astype(str).to_numpy()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

numerical_values_tensor = torch.tensor(numerical_values, dtype=torch.float32).to(device)
categorical_indices_tensor = torch.tensor(categorical_indices, dtype=torch.long).to(device)

# safer normalization for numerical values (ignore -1 missing)
numerical_values_norm = numerical_values_tensor.clone()
for j in range(numerical_values_norm.shape[1]):
    mask = numerical_values_norm[:, j] != -1
    if mask.any():
        mean_j = numerical_values_norm[mask, j].mean()
        std_j = numerical_values_norm[mask, j].std()
        numerical_values_norm[mask, j] = (numerical_values_norm[mask, j] - mean_j) / (std_j + 1e-8)

# -------------------------------------------------
# Updated Memory Bank Population
# -------------------------------------------------
num_total_attrs = normalized_embeddings.shape[1]
PerAttributeMemoryBank1 = PerAttributeMemoryBank(
    num_attributes=num_total_attrs,
    d_h=normalized_embeddings.shape[2]
)

num_txt = len(textual_columns)
num_cat = len(categorical_columns)
num_num = len(numerical_columns)

for n in range(len(normalized_embeddings)):
    for attr_idx in range(num_total_attrs):
        emb = normalized_embeddings[n][attr_idx]

        if emb.abs().sum().item() <= 1e-6:
            continue

        modality = modalities[attr_idx]

        if modality == "txt":
            PerAttributeMemoryBank1.add(
                attr_id=attr_idx,
                embedding=emb,
                modality="txt",
                value=None
            )

        elif modality == "cat":
            local_cat_idx = attr_idx - num_txt
            cat_val = categorical_indices[n, local_cat_idx]

            if cat_val == -1:
                continue

            PerAttributeMemoryBank1.add(
                attr_id=attr_idx,
                embedding=emb,
                modality="cat",
                value=int(cat_val)
            )

        elif modality == "num":
            local_num_idx = attr_idx - num_txt - num_cat
            num_val = numerical_values[n, local_num_idx]

            if num_val == -1:
                continue

            PerAttributeMemoryBank1.add(
                attr_id=attr_idx,
                embedding=emb,
                modality="num",
                value=float(num_val)
            )

# -------------------------------------------------
# Edge construction
# -------------------------------------------------
def build_modality_based_edge_index(normalized_embeddings, modality_types, top_k=5):
    N, M, D = normalized_embeddings.shape

    attr_embeddings = []
    for m in range(M):
        emb_m = normalized_embeddings[:, m, :]
        non_zero_mask = (emb_m.abs().sum(dim=1) != 0)
        if non_zero_mask.any():
            mean_emb = emb_m[non_zero_mask].mean(dim=0)
        else:
            mean_emb = torch.zeros(D, dtype=normalized_embeddings.dtype)
        attr_embeddings.append(mean_emb)

    attr_embeddings = torch.stack(attr_embeddings, dim=0)
    attr_embeddings = F.normalize(attr_embeddings, dim=1)

    modality_groups = defaultdict(list)
    for i, mod in enumerate(modality_types):
        modality_groups[mod].append(i)

    edges = []
    for mod, indices in modality_groups.items():
        if len(indices) < 2:
            continue

        mod_embs = attr_embeddings[indices]
        sim_matrix = torch.matmul(mod_embs, mod_embs.T)

        for i, src_idx in enumerate(indices):
            sim_scores = sim_matrix[i]
            topk = torch.topk(sim_scores, k=min(top_k + 1, len(indices))).indices.tolist()

            for tgt_local in topk:
                tgt_idx = indices[tgt_local]
                if src_idx != tgt_idx:
                    edges.append((src_idx, tgt_idx))

    edges += [(j, i) for (i, j) in edges]
    edges = list(set(edges))

    edge_index = torch.tensor(edges, dtype=torch.long).T
    return edge_index

edge_index = build_modality_based_edge_index(
    normalized_embeddings=normalized_embeddings,
    modality_types=modalities,
    top_k=5
).to(device)

# -------------------------------------------------
# Instantiate model
# -------------------------------------------------
model = RAGNet(
    d_llm=384,
    d_num=32,
    d_cat=16,
    d_shared=384,
    num_numerical_attrs=len(numerical_columns),
    num_categorical_attrs=len(categorical_columns),
    num_classes_per_cat_attr=num_classes_per_cat_attr,
    vocab_size=tokenizer.vocab_size,
    decoder_dim=384,
    max_text_len=64,
    loss_fn=loss_fn,
    PerAttributeMemoryBank1=PerAttributeMemoryBank1
).to(device)
####################
# -------------------------------------------------
# Load saved model from last run if it exists
# -------------------------------------------------
save_model_path = "trained_model_fixed_split.pt"

#if os.path.exists(save_model_path):
#    print(f"🔄 Loading saved model from: {save_model_path}")
#    checkpoint = torch.load(save_model_path, map_location=device)

    # supports either full checkpoint dict or plain state_dict
#    if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
#        model.load_state_dict(checkpoint["model_state_dict"])
#        print("✅ Loaded model_state_dict from checkpoint")
#    else:
#        model.load_state_dict(checkpoint)
#        print("✅ Loaded plain state_dict")
#else:
#    print("ℹ️ No saved model found. Training from scratch.")
####################
# -------------------------------------------------
# Embeddings tensor for training
# -------------------------------------------------
if isinstance(normalized_embeddings, np.ndarray):
    normalized_embeddings_tensor = torch.tensor(normalized_embeddings, dtype=torch.float32).to(device)
else:
    normalized_embeddings_tensor = normalized_embeddings.to(device)

all_attribute_names = textual_columns + categorical_columns + numerical_columns
modalities = ['txt'] * len(textual_columns) + ['cat'] * len(categorical_columns) + ['num'] * len(numerical_columns)

# -------------------------------------------------


📄 Text Attr 0: 755 observed, Avg. Length: 9.94
📄 Text Attr 1: 375 observed, Avg. Length: 9.82
📄 Text Attr 2: 447 observed, Avg. Length: 8.95


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [ ]:
# Continue training
# -------------------------------------------------
model, train_idx, test_idx = train_with_fixed_split(
    model=model,
    normalized_embeddings=normalized_embeddings,
    ground_truth=ground_truth,
    numerical_values=numerical_values,
    categorical_indices=categorical_indices,
    modalities=modalities,
    edge_index=edge_index,
    device=device,
    split_json_path="fixed_eval_setup_test_ratio_multi_13cat_9num_1/fixed_setup.json",
    epochs=50,
    batch_size=16,
    lr=1e-4,
    save_model_path=save_model_path
)

Loaded split from fixed setup: fixed_eval_setup_test_ratio_multi_13cat_9num_1/fixed_setup.json

===== TRAINING WITH FIXED SPLIT =====
Train size: 528
Test size: 227
[Epoch 1/50] Avg Train Loss: 45.6908
[Epoch 2/50] Avg Train Loss: 28.1144
[Epoch 3/50] Avg Train Loss: 23.5251
[Epoch 4/50] Avg Train Loss: 20.9698
[Epoch 5/50] Avg Train Loss: 19.1880
[Epoch 6/50] Avg Train Loss: 17.7708
[Epoch 7/50] Avg Train Loss: 16.7585
[Epoch 8/50] Avg Train Loss: 16.0148
[Epoch 9/50] Avg Train Loss: 15.4237
[Epoch 10/50] Avg Train Loss: 14.9138
[Epoch 11/50] Avg Train Loss: 14.4923
[Epoch 12/50] Avg Train Loss: 14.2009
[Epoch 13/50] Avg Train Loss: 13.9593
[Epoch 14/50] Avg Train Loss: 13.7418
[Epoch 15/50] Avg Train Loss: 13.5273
[Epoch 16/50] Avg Train Loss: 13.3592
[Epoch 17/50] Avg Train Loss: 13.1453
[Epoch 18/50] Avg Train Loss: 13.0071
[Epoch 19/50] Avg Train Loss: 12.8978
[Epoch 20/50] Avg Train Loss: 12.7404
[Epoch 21/50] Avg Train Loss: 12.6383
[Epoch 22/50] Avg Train Loss: 12.5608
[Epoch 2

In [ ]:
import os
import json
import random
import numpy as np
import torch
from collections import defaultdict
from sklearn.metrics import f1_score, accuracy_score, mean_squared_error, mean_absolute_error
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction

# ============================================================
# EVALUATION HELPERS
# ============================================================

def evaluate_per_sample_with_uploaded_mask(
    model,
    normalized_embeddings,
    ground_truth,
    modalities,
    edge_index,
    numerical_values,
    categorical_indices,
    tokenizer,
    mask_file_path
):
    """
    Evaluate model using ONE uploaded fixed mask file.

    Metrics style matches the previous function:
      Numerical:
        - num_mae_weighted = sum_i w_i * MAE_i
        - num_mse_weighted = sum_i w_i * MSE_i
        - num_mae_macro_attr
        - num_mse_macro_attr

      Categorical:
        - cat_f1_weighted = sum_i w_i * F1_i
        - cat_acc_weighted = sum_i w_i * ACC_i
        - cat_f1_macro_attr
        - cat_acc_macro_attr

      Text:
        - txt_token_acc_weighted = sum_i w_i * token_acc_i
        - txt_bleu_weighted = sum_i w_i * bleu_i
        - txt_token_acc_macro_attr
        - txt_bleu_macro_attr
    """
    model.eval()

    with open(mask_file_path, "r") as f:
        mask_payload = json.load(f)

    fixed_masks = mask_payload["masks"]
    test_idx = [int(x) for x in mask_payload["test_idx"]]

    total_eval_loss = 0.0
    total_valid_count = 0

    num_targets_by_attr = defaultdict(list)
    num_preds_by_attr = defaultdict(list)

    cat_targets_by_attr = defaultdict(list)
    cat_preds_by_attr = defaultdict(list)

    txt_token_correct_by_attr = defaultdict(int)
    txt_token_total_by_attr = defaultdict(int)
    txt_sample_count_by_attr = defaultdict(int)

    bleu_references_by_attr = defaultdict(list)
    bleu_hypotheses_by_attr = defaultdict(list)

    smooth = SmoothingFunction().method4

    with torch.no_grad():
        for sample_idx in test_idx:
            sample_idx_str = str(int(sample_idx))
            if sample_idx_str not in fixed_masks:
                continue

            masked_attr_indices = fixed_masks[sample_idx_str]
            if len(masked_attr_indices) == 0:
                continue

            emb = normalized_embeddings[sample_idx].clone()
            gt = ground_truth[sample_idx]

            missing_mask = []
            for x in gt:
                if isinstance(x, str):
                    x_str = x.strip()
                    missing_mask.append(
                        x_str == "" or x_str == "-1" or x_str.lower() == "nan"
                    )
                else:
                    try:
                        missing_mask.append(float(x) == -1)
                    except Exception:
                        missing_mask.append(False)

            missing_mask = torch.tensor(
                missing_mask, dtype=torch.bool, device=emb.device
            )

            emb_masked = emb.clone()
            mask_copy = missing_mask.clone()

            for masked_attr_idx in masked_attr_indices:
                emb_masked[masked_attr_idx] = 0.0
                mask_copy[masked_attr_idx] = True

            loss, outputs = model(
                llm_embeddings=emb_masked,
                raw_numericals=numerical_values[sample_idx],
                cat_indices=categorical_indices[sample_idx],
                missing_mask=mask_copy,
                edge_index=edge_index,
                modality_types=modalities,
                targets=gt,
                attr_indices=list(range(len(modalities))),
                mode='test'
            )

            if loss is not None and torch.is_tensor(loss) and not torch.isnan(loss) and not torch.isinf(loss):
                total_eval_loss += loss.item()

            for masked_attr_idx in masked_attr_indices:
                mod = modalities[masked_attr_idx]
                gt_i = gt[masked_attr_idx]
                out_i = outputs[masked_attr_idx]

                # -----------------------------
                # Numerical
                # -----------------------------
                if mod == 'num':
                    try:
                        gt_val = float(gt_i)
                    except Exception:
                        continue

                    if gt_val == -1:
                        continue

                    if torch.is_tensor(out_i):
                        pred_val = out_i.squeeze().item()
                    else:
                        pred_val = float(out_i)

                    num_targets_by_attr[masked_attr_idx].append(gt_val)
                    num_preds_by_attr[masked_attr_idx].append(pred_val)
                    total_valid_count += 1

                # -----------------------------
                # Categorical
                # -----------------------------
                elif mod == 'cat':
                    try:
                        gt_val = int(gt_i)
                    except Exception:
                        continue

                    if gt_val == -1:
                        continue

                    if torch.is_tensor(out_i):
                        if out_i.dim() == 2:
                            pred_val = out_i.argmax(dim=1).item()
                        else:
                            pred_val = out_i.argmax().item()
                    else:
                        pred_val = int(out_i)

                    cat_targets_by_attr[masked_attr_idx].append(gt_val)
                    cat_preds_by_attr[masked_attr_idx].append(pred_val)
                    total_valid_count += 1

                # -----------------------------
                # Text
                # -----------------------------
                elif mod == 'txt':
                    if isinstance(gt_i, str):
                        gt_i_str = gt_i.strip()
                        if len(gt_i_str) == 0 or gt_i_str == "-1" or gt_i_str.lower() == "nan":
                            continue
                        gt_ids = tokenizer.encode(gt_i, add_special_tokens=False)
                    elif isinstance(gt_i, torch.Tensor):
                        gt_ids = gt_i.view(-1).tolist()
                    elif isinstance(gt_i, (list, tuple)):
                        gt_ids = list(gt_i)
                    else:
                        try:
                            gt_ids = [int(gt_i)]
                        except Exception:
                            continue

                    if not torch.is_tensor(out_i):
                        continue

                    if out_i.dim() == 3:
                        pred_ids = out_i.argmax(dim=-1).squeeze(0).tolist()
                    elif out_i.dim() == 2:
                        pred_ids = out_i.argmax(dim=-1).tolist()
                    else:
                        raise ValueError(f"Unexpected text output shape: {out_i.shape}")

                    ignore_tokens = {0, 101, 102}
                    gt_ids = [int(t) for t in gt_ids if int(t) not in ignore_tokens]
                    pred_ids = [int(t) for t in pred_ids if int(t) not in ignore_tokens]

                    min_len = min(len(gt_ids), len(pred_ids))
                    if min_len > 0:
                        txt_token_correct_by_attr[masked_attr_idx] += sum(
                            g == p for g, p in zip(gt_ids[:min_len], pred_ids[:min_len])
                        )
                        txt_token_total_by_attr[masked_attr_idx] += min_len

                    ref_sentence = tokenizer.decode(gt_ids, skip_special_tokens=True)
                    hyp_sentence = tokenizer.decode(pred_ids, skip_special_tokens=True)

                    ref_words = ref_sentence.split()
                    hyp_words = hyp_sentence.split()

                    if len(ref_words) > 0 and len(hyp_words) > 0:
                        bleu_references_by_attr[masked_attr_idx].append([ref_words])
                        bleu_hypotheses_by_attr[masked_attr_idx].append(hyp_words)
                        txt_sample_count_by_attr[masked_attr_idx] += 1

                    total_valid_count += 1

    results = {}

    # -----------------------------
    # Numerical
    # -----------------------------
    num_mae_per_attr = {}
    num_mse_per_attr = {}
    num_valid_per_attr = {}

    for attr_idx in sorted(num_targets_by_attr.keys()):
        y_true = num_targets_by_attr[attr_idx]
        y_pred = num_preds_by_attr[attr_idx]

        if len(y_true) == 0:
            continue

        num_mae_per_attr[attr_idx] = mean_absolute_error(y_true, y_pred)
        num_mse_per_attr[attr_idx] = mean_squared_error(y_true, y_pred)
        num_valid_per_attr[attr_idx] = len(y_true)

    if len(num_valid_per_attr) > 0:
        total_num_valid = sum(num_valid_per_attr.values())

        results["num_mae_weighted"] = sum(
            (num_valid_per_attr[a] / total_num_valid) * num_mae_per_attr[a]
            for a in num_mae_per_attr
        )
        results["num_mse_weighted"] = sum(
            (num_valid_per_attr[a] / total_num_valid) * num_mse_per_attr[a]
            for a in num_mse_per_attr
        )
        results["num_mae_macro_attr"] = np.mean(list(num_mae_per_attr.values()))
        results["num_mse_macro_attr"] = np.mean(list(num_mse_per_attr.values()))
        results["num_mae_per_attr"] = dict(num_mae_per_attr)
        results["num_mse_per_attr"] = dict(num_mse_per_attr)
        results["num_valid_per_attr"] = dict(num_valid_per_attr)
    else:
        results["num_mae_weighted"] = 0.0
        results["num_mse_weighted"] = 0.0
        results["num_mae_macro_attr"] = 0.0
        results["num_mse_macro_attr"] = 0.0
        results["num_mae_per_attr"] = {}
        results["num_mse_per_attr"] = {}
        results["num_valid_per_attr"] = {}

    # -----------------------------
    # Categorical
    # -----------------------------
    cat_f1_per_attr = {}
    cat_acc_per_attr = {}
    cat_valid_per_attr = {}

    for attr_idx in sorted(cat_targets_by_attr.keys()):
        y_true = cat_targets_by_attr[attr_idx]
        y_pred = cat_preds_by_attr[attr_idx]

        if len(y_true) == 0:
            continue

        cat_f1_per_attr[attr_idx] = f1_score(
            y_true, y_pred, average="weighted", zero_division=0
        )
        cat_acc_per_attr[attr_idx] = accuracy_score(y_true, y_pred)
        cat_valid_per_attr[attr_idx] = len(y_true)

    if len(cat_valid_per_attr) > 0:
        total_cat_valid = sum(cat_valid_per_attr.values())

        results["cat_f1_weighted"] = sum(
            (cat_valid_per_attr[a] / total_cat_valid) * cat_f1_per_attr[a]
            for a in cat_f1_per_attr
        )
        results["cat_acc_weighted"] = sum(
            (cat_valid_per_attr[a] / total_cat_valid) * cat_acc_per_attr[a]
            for a in cat_acc_per_attr
        )
        results["cat_f1_macro_attr"] = np.mean(list(cat_f1_per_attr.values()))
        results["cat_acc_macro_attr"] = np.mean(list(cat_acc_per_attr.values()))
        results["cat_f1_per_attr"] = dict(cat_f1_per_attr)
        results["cat_acc_per_attr"] = dict(cat_acc_per_attr)
        results["cat_valid_per_attr"] = dict(cat_valid_per_attr)
    else:
        results["cat_f1_weighted"] = 0.0
        results["cat_acc_weighted"] = 0.0
        results["cat_f1_macro_attr"] = 0.0
        results["cat_acc_macro_attr"] = 0.0
        results["cat_f1_per_attr"] = {}
        results["cat_acc_per_attr"] = {}
        results["cat_valid_per_attr"] = {}

    # -----------------------------
    # Text
    # -----------------------------
    txt_token_acc_per_attr = {}
    txt_bleu_per_attr = {}
    txt_valid_per_attr = {}

    txt_attr_ids = sorted(
        set(list(txt_token_total_by_attr.keys()) + list(bleu_references_by_attr.keys()))
    )

    for attr_idx in txt_attr_ids:
        token_total = txt_token_total_by_attr.get(attr_idx, 0)
        refs = bleu_references_by_attr.get(attr_idx, [])
        hyps = bleu_hypotheses_by_attr.get(attr_idx, [])

        txt_token_acc_per_attr[attr_idx] = (
            txt_token_correct_by_attr[attr_idx] / token_total if token_total > 0 else 0.0
        )

        txt_bleu_per_attr[attr_idx] = (
            corpus_bleu(refs, hyps, smoothing_function=smooth) if len(refs) > 0 else 0.0
        )

        txt_valid_per_attr[attr_idx] = txt_sample_count_by_attr.get(attr_idx, 0)

    if len(txt_valid_per_attr) > 0 and sum(txt_valid_per_attr.values()) > 0:
        total_txt_valid = sum(txt_valid_per_attr.values())

        results["txt_token_acc_weighted"] = sum(
            (txt_valid_per_attr[a] / total_txt_valid) * txt_token_acc_per_attr[a]
            for a in txt_token_acc_per_attr
        )
        results["txt_bleu_weighted"] = sum(
            (txt_valid_per_attr[a] / total_txt_valid) * txt_bleu_per_attr[a]
            for a in txt_bleu_per_attr
        )
        results["txt_token_acc_macro_attr"] = np.mean(list(txt_token_acc_per_attr.values()))
        results["txt_bleu_macro_attr"] = np.mean(list(txt_bleu_per_attr.values()))
        results["txt_token_acc_per_attr"] = dict(txt_token_acc_per_attr)
        results["txt_bleu_per_attr"] = dict(txt_bleu_per_attr)
        results["txt_valid_per_attr"] = dict(txt_valid_per_attr)
    else:
        results["txt_token_acc_weighted"] = 0.0
        results["txt_bleu_weighted"] = 0.0
        results["txt_token_acc_macro_attr"] = 0.0
        results["txt_bleu_macro_attr"] = 0.0
        results["txt_token_acc_per_attr"] = {}
        results["txt_bleu_per_attr"] = {}
        results["txt_valid_per_attr"] = {}

    results["total_loss"] = total_eval_loss / max(total_valid_count, 1)
    results["total_valid_count"] = total_valid_count
    results["mask_file_path"] = mask_file_path
    results["missing_scenario"] = mask_payload.get("missing_scenario", "unknown")

    print(results)
    return results

In [ ]:
def average_results(results_list):
    metric_keys = [
        "num_mae_weighted",
        "num_mse_weighted",
        "num_mae_macro_attr",
        "num_mse_macro_attr",
        "cat_f1_weighted_attr_weighted",
        "cat_f1_macro_attr_weighted",
        "cat_acc_weighted",
        "cat_f1_weighted_macro_attr",
        "cat_f1_macro_macro_attr",
        "cat_acc_macro_attr",
        "txt_token_acc_weighted",
        "txt_bleu_weighted",
        "txt_token_acc_macro_attr",
        "txt_bleu_macro_attr",
        "total_loss",
    ]

    avg = {}
    std = {}

    for key in metric_keys:
        vals = [r[key] for r in results_list if key in r and r[key] is not None]
        if len(vals) == 0:
            avg[key] = 0.0
            std[key + "_std"] = 0.0
        else:
            avg[key] = float(np.mean(vals))
            std[key + "_std"] = float(np.std(vals))

    avg.update(std)
    return avg

In [ ]:
def evaluate_model_with_masks(
    model,
    normalized_embeddings,
    ground_truth,
    modalities,
    edge_index,
    numerical_values,
    categorical_indices,
    tokenizer,
    mask_file_paths
):
    """
    Accepts:
      - one mask path as string
      - or list of mask paths

    Returns:
      - single result if one mask
      - average result if many masks
    """
    def print_f1_per_class(res):
        if "per_class_f1" in res:
            print("F1 per class:")
            for i, f1 in enumerate(res["per_class_f1"]):
                print(f"  Class {i}: {f1:.4f}")

    if isinstance(mask_file_paths, str):
        res = evaluate_per_sample_with_uploaded_mask(
            model=model,
            normalized_embeddings=normalized_embeddings,
            ground_truth=ground_truth,
            modalities=modalities,
            edge_index=edge_index,
            numerical_values=numerical_values,
            categorical_indices=categorical_indices,
            tokenizer=tokenizer,
            mask_file_path=mask_file_paths
        )
        print_f1_per_class(res)
        return res

    all_results = []
    for path in mask_file_paths:
        print(f"\nEvaluating mask: {path}")
        res = evaluate_per_sample_with_uploaded_mask(
            model=model,
            normalized_embeddings=normalized_embeddings,
            ground_truth=ground_truth,
            modalities=modalities,
            edge_index=edge_index,
            numerical_values=numerical_values,
            categorical_indices=categorical_indices,
            tokenizer=tokenizer,
            mask_file_path=path
        )

        print_f1_per_class(res)
        all_results.append(res)

    avg = average_results(all_results)

    print("\n===== AVERAGE OVER MASKS =====")
    print(avg)

    if "per_class_f1" in avg:
        print_f1_per_class(avg)

    return avg

In [ ]:
def evaluate_all_scenarios(
    model,
    normalized_embeddings,
    ground_truth,
    modalities,
    edge_index,
    numerical_values,
    categorical_indices,
    tokenizer,
    masks_root_dir="fixed_eval_setup/masks",
    scenarios=("random", "mcar", "mar", "mnar"),
    num_masks=10
):
    all_scenario_results = {}

    for scenario in scenarios:
        mask_paths = [
            os.path.join(masks_root_dir, scenario, f"mask_{i}.json")
            for i in range(num_masks)
        ]

        print(f"\n================ {scenario.upper()} ================\n")
        scenario_avg = evaluate_model_with_masks(
            model=model,
            normalized_embeddings=normalized_embeddings,
            ground_truth=ground_truth,
            modalities=modalities,
            edge_index=edge_index,
            numerical_values=numerical_values,
            categorical_indices=categorical_indices,
            tokenizer=tokenizer,
            mask_file_paths=mask_paths
        )

        all_scenario_results[scenario] = scenario_avg

    print("\n===== FINAL AVERAGE RESULTS FOR ALL SCENARIOS =====")
    for scenario, metrics in all_scenario_results.items():
        print(f"\nScenario: {scenario}")
        for k, v in metrics.items():
            print(f"  {k}: {v}")

    return all_scenario_results

In [ ]:
all_results = {}

for ratio in [0.1, 0.2, 0.3, 0.4, 0.5]:
    ratio_str = f"{ratio:.1f}"

    mcar_file_paths = [
        f"fixed_eval_setup_test_ratio_multi_13cat_9num_1/masks/mcar/ratio_{ratio_str}/mask_{i}.json"
        for i in range(5)
    ]

    results = evaluate_model_with_masks(
        model=model,
        normalized_embeddings=normalized_embeddings,
        ground_truth=ground_truth,
        modalities=modalities,
        edge_index=edge_index,
        numerical_values=numerical_values,
        categorical_indices=categorical_indices,
        tokenizer=tokenizer,
        mask_file_paths=mcar_file_paths
    )

    all_results[ratio_str] = results


Evaluating mask: fixed_eval_setup_test_ratio_multi_13cat_9num_1/masks/mcar/ratio_0.1/mask_0.json
{'num_mae_weighted': 0.7560707647514215, 'num_mse_weighted': 1.0894416626502657, 'num_mae_macro_attr': np.float64(0.485040011825429), 'num_mse_macro_attr': np.float64(0.556550877831898), 'num_mae_per_attr': {16: 0.8681444879992538, 17: 0.13653490724537373, 18: 0.322221112110247, 20: 0.03076491523829311, 21: 0.5613619213089601, 22: 0.5978486841780765, 23: 1.095014826590261, 24: 0.2684292399329664}, 'num_mse_per_attr': {16: 1.5487998981637545, 17: 0.019339754355630134, 18: 0.11161683250472625, 20: 0.0015900854615000341, 21: 0.34729028224687863, 22: 0.5898436343380263, 23: 1.6159133832590915, 24: 0.2180131523255767}, 'num_valid_per_attr': {16: 31, 17: 4, 18: 3, 20: 4, 21: 3, 22: 7, 23: 72, 24: 40}, 'cat_f1_weighted': 0.5286328509656073, 'cat_acc_weighted': 0.6297709923664122, 'cat_f1_macro_attr': np.float64(0.5814779171340742), 'cat_acc_macro_attr': np.float64(0.6731349405969684), 'cat_f1_per

In [ ]:
all_results = {}

for ratio in [0.1, 0.2, 0.3, 0.4, 0.5]:
    ratio_str = f"{ratio:.1f}"

    mcar_file_paths = [
        f"fixed_eval_setup_test_ratio_multi_13cat_9num_1/masks/mar/ratio_{ratio_str}/mask_{i}.json"
        for i in range(5)
    ]

    results = evaluate_model_with_masks(
        model=model,
        normalized_embeddings=normalized_embeddings,
        ground_truth=ground_truth,
        modalities=modalities,
        edge_index=edge_index,
        numerical_values=numerical_values,
        categorical_indices=categorical_indices,
        tokenizer=tokenizer,
        mask_file_paths=mcar_file_paths
    )

    all_results[ratio_str] = results


Evaluating mask: fixed_eval_setup_test_ratio_multi_13cat_9num_1/masks/mar/ratio_0.1/mask_0.json
{'num_mae_weighted': 0.5816080984262972, 'num_mse_weighted': 0.7761522032031931, 'num_mae_macro_attr': np.float64(0.6346835891928156), 'num_mse_macro_attr': np.float64(0.8990944597339278), 'num_mae_per_attr': {16: 1.1808325915276399, 17: 0.13829969052191673, 20: 0.057591764254164825, 21: 1.056477638108038, 22: 0.760251396204862, 23: 1.0261545263072613, 24: 0.22317751742582603}, 'num_mse_per_attr': {16: 2.1781496878815982, 17: 0.01956594574748052, 20: 0.00488538760335646, 21: 1.6084421943058136, 22: 0.9549911753079876, 23: 1.2922416577952862, 24: 0.23538516949597152}, 'num_valid_per_attr': {16: 15, 17: 3, 20: 6, 21: 5, 22: 7, 23: 47, 24: 81}, 'cat_f1_weighted': 0.4899297460674911, 'cat_acc_weighted': 0.583969465648855, 'cat_f1_macro_attr': np.float64(0.4991751956740725), 'cat_acc_macro_attr': np.float64(0.583654571343578), 'cat_f1_per_attr': {4: 0.6428571428571428, 5: 0.11145510835913312, 6:

In [ ]:
all_results = {}

for ratio in [0.1, 0.2, 0.3, 0.4, 0.5]:
    ratio_str = f"{ratio:.1f}"

    mcar_file_paths = [
        f"fixed_eval_setup_test_ratio_multi_13cat_9num_1/masks/mnar/ratio_{ratio_str}/mask_{i}.json"
        for i in range(5)
    ]

    results = evaluate_model_with_masks(
        model=model,
        normalized_embeddings=normalized_embeddings,
        ground_truth=ground_truth,
        modalities=modalities,
        edge_index=edge_index,
        numerical_values=numerical_values,
        categorical_indices=categorical_indices,
        tokenizer=tokenizer,
        mask_file_paths=mcar_file_paths
    )

    all_results[ratio_str] = results


Evaluating mask: fixed_eval_setup_test_ratio_multi_13cat_9num_1/masks/mnar/ratio_0.1/mask_0.json
{'num_mae_weighted': 0.6863231333996895, 'num_mse_weighted': 1.0816419479712307, 'num_mae_macro_attr': np.float64(0.6334737454932424), 'num_mse_macro_attr': np.float64(0.9031060325306872), 'num_mae_per_attr': {16: 0.9216562507048007, 17: 0.10286408729286445, 18: 1.1093011490379736, 19: 0.36802278378719455, 20: 0.026346441215284844, 21: 1.08773972973741, 22: 0.6962011696180904, 23: 1.1520412755350902, 24: 0.23709082251047273}, 'num_mse_per_attr': {16: 1.727846074940548, 17: 0.012025280618567662, 18: 2.018309296444843, 19: 0.1362370627622049, 20: 0.00069413496471046, 21: 1.531887952721905, 22: 0.6450999866998817, 23: 1.7540089069905385, 24: 0.30184559663298455}, 'num_valid_per_attr': {16: 46, 17: 6, 18: 2, 19: 3, 20: 1, 21: 9, 22: 7, 23: 33, 24: 57}, 'cat_f1_weighted': 0.39996399785914827, 'cat_acc_weighted': 0.4465648854961832, 'cat_f1_macro_attr': np.float64(0.312119499536265), 'cat_acc_ma